# 🎯 Fine-tune Kronos trên dữ liệu của bạn (US30 / USTEC M30)

Huấn luyện lại Kronos-small trên đúng mã của bạn để **tạo edge** mà model gốc chưa có, rồi **đo lại edge out-of-sample** so với model gốc.

**Bắt buộc GPU:** Runtime → Change runtime type → **T4 GPU**. (Fine-tune trên CPU là bất khả thi.)

> Quy trình: clone → cài đặt → fine-tune (tokenizer → predictor) → đo edge model mới vs gốc trên phần dữ liệu CHƯA train.

## Bước 1 — Kiểm tra GPU (bắt buộc)

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ CHƯA bật GPU! Runtime > Change runtime type > T4 GPU, rồi chạy lại.'
print('✅ GPU:', torch.cuda.get_device_name(0))

## Bước 2 — Tải code & cài đặt

Repo đã public nên clone không cần token. Chỉ cài thêm vài thư viện Kronos cần (torch/pandas đã có sẵn trên Colab).

In [ ]:
import os, sys
os.chdir('/content')
if not os.path.isdir('/content/Kronoss/.git'):
    !git clone --quiet --branch claude/affectionate-fermi-b956vh https://github.com/nguyenlam19122/Kronoss.git
os.chdir('/content/Kronoss'); sys.path.insert(0, '/content/Kronoss')
!pip install -q einops huggingface_hub safetensors pyyaml
import trading; print('✅ Sẵn sàng — cwd:', os.getcwd())

## Bước 3 — Chọn mã & fine-tune

Đổi `SYM` thành `US30` hoặc `USTEC`. Quá trình gồm 2 giai đoạn (tokenizer → predictor), ~30–90 phút trên T4.

Config nằm ở `finetune_csv/configs/config_{SYM}_M30.yaml` (lookback 256, predict 16, train 70% / val 15% / test 15%). Muốn chỉnh epoch/learning-rate thì sửa file đó.

In [ ]:
SYM = 'US30'   # hoặc 'USTEC'
import os
os.chdir('/content/Kronoss/finetune_csv')
!python train_sequential.py --config configs/config_{SYM}_M30.yaml
os.chdir('/content/Kronoss')
print('Model đã fine-tune lưu ở: finetune_csv/finetuned/%s_M30/' % SYM)

## Bước 4 — Đo edge: model FINE-TUNED vs GỐC (out-of-sample)

So sánh trên **phần đuôi dữ liệu chưa train** (last 2000 nến). Nếu fine-tune có tác dụng, model mới phải có **accuracy / IC cao hơn** model gốc.

In [ ]:
DATA = f'trading/mydata/{SYM}_M30.csv'
FT_MODEL = f'finetune_csv/finetuned/{SYM}_M30/basemodel/best_model'
FT_TOK   = f'finetune_csv/finetuned/{SYM}_M30/tokenizer/best_model'
COMMON = '--lookback 256 --pred-len 6 --horizon 3 --signal-every 8 --last-n 2000 --ensemble 5'

base_cmd = ('python -m trading.measure_edge_all --predictor kronos '
            '--model NeoQuasar/Kronos-small --tokenizer NeoQuasar/Kronos-Tokenizer-base '
            f'{COMMON} --csv {DATA}')
ft_cmd = ('python -m trading.measure_edge_all --predictor kronos '
          f'--model {FT_MODEL} --tokenizer {FT_TOK} {COMMON} --csv {DATA}')

print('===== MODEL GỐC (base Kronos-small) =====')
!{base_cmd}
print('\n===== MODEL FINE-TUNED =====')
!{ft_cmd}

## Bước 5 — Diễn giải & bước tiếp

- **Fine-tune thành công** = model mới có **accuracy cao hơn rõ** (vd 53–55%+), IC/RankIC dương hơn model gốc, trên phần out-of-sample.
- **Không cải thiện?** Thử: tăng `basemodel_epochs`, đổi `predictor_learning_rate` (vd 5e-5), giảm `predict_window`, hoặc dùng nhiều dữ liệu hơn (mã có lịch sử dài như XAUUSD 87k nến).
- **Cẩn thận overfit:** đây là dữ liệu nhỏ (~11k nến). Edge out-of-sample mới đáng tin; đừng nhìn loss train.
- **Có edge rồi?** Quay lại notebook `trading/Kronos_Trading_Colab.ipynb`, đặt `MODEL_NAME` = đường dẫn model fine-tuned để backtest có SL/TP/sizing đầy đủ.

> Lưu ý: muốn chỉ fine-tune predictor (nhanh hơn, an toàn hơn với dữ liệu nhỏ), thêm `--skip-tokenizer` vào lệnh ở Bước 3 — nhưng khi đó cần tokenizer gốc; hỏi mình để chỉnh config cho đúng.